# 🧱 Part 4：从零构建 Mini GPT — Self-Attention + Transformer Block

> **前情回顾**：有了 Tokenizer（文本→数字），有了 Embedding + Position（数字→向量）。
>
> **现在的核心问题**：这组向量怎么变成对下一个 token 的预测？
>
> 本 Part 目标：**手写 Self-Attention，理解 Q/K/V 到底是什么，然后搭出完整的 Transformer Block。**

**这是整个课程最难的部分，但也是最重要的。** 每一步都会用非常小的例子演算，保证你能跟上。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 0. 在 Attention 之前：NLP 的三大任务和序列建模的困境

> 要理解 Attention 为什么是革命性的，你得先知道它解决了什么问题。

#### NLP 任务的三大分类

所有自然语言处理任务，都可以被归入这三类。这个框架贯穿后续所有内容：

| 任务类型 | 说明 | 例子 | 输入/输出 |
|---------|------|------|-----------|
| **分类** | 给整段文本一个标签 | 影评情感分析 | "太好看了" → 正面 |
| **序列标注** | 给每个词打一个标签 | 命名实体识别 | "马云在杭州" → [人名, 其他, 地名] |
| **生成** | 给定输入，产出新文本 | 翻译、摘要、对话 | "Hello" → "你好" |

大语言模型（GPT 系列）属于**生成**类——它们最强的能力就是「接话」。
而 BERT 类模型擅长前两类——理解和标注。后面我们会看到架构选择如何决定了模型擅长什么。

#### 在 Transformer 之前：RNN/LSTM 怎么做序列建模？

语言的本质是序列——词和词之间有顺序依赖。比如：
```
"把 猫 放在 垫子 上"  vs  "把 垫子 放在 猫 上"
```
同样的词，顺序不同，意思完全不同。

RNN 的核心思想：**用隐藏状态 h_t 把之前的信息一步步传下去**
```
x₁ → [RNN] → h₁ → [RNN] → h₂ → [RNN] → h₃ → ...
        ↑               ↑               ↑
     输入词1    h₁+输入词2    h₂+输入词3
```
每一个时间步的 $h_t$ 包含了前面所有词的信息（理论上）。

#### RNN 的致命弱点：长程依赖 & 梯度消失

当序列很长时（比如一段 500 词的文本），RNN 出问题了：

1. **信息衰减**：第 1 个词的信息经过 500 次传递后，到第 500 步时已经几乎消失了——就像传话游戏，传了 500 个人后第一句话已经面目全非
2. **梯度消失**：反向传播时，梯度要从最后一步一直乘到第一步。每一步乘一个小于 1 的数，乘 500 次后梯度几乎为 0——早期的参数根本学不到东西
3. **串行计算**：第 t 步必须等第 t-1 步算完，无法并行——训练很慢

下面用一个具体的代码演示这个「梯度消失」现象：

In [ ]:
# ============================================================
# 演示 RNN 的梯度消失问题
# ============================================================
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

# 模拟一个简化版 RNN：每一步的隐藏状态 = tanh(W_h * h_prev + W_x * x)
# 我们关心：第 0 步的输入对第 t 步的输出有多大的梯度

seq_len = 100
hidden_dim = 1  # 用 1 维来可视化

# 为了演示梯度消失，用 tanh 作为激活函数
# tanh 的导数最大为 1.0，在饱和区接近 0
W_h = torch.tensor([[0.8]])  # 隐藏状态权重（<1 导致梯度消失，>1 导致梯度爆炸）
W_x = torch.tensor([[0.5]])  # 输入权重

x = torch.randn(seq_len, 1, requires_grad=True)

# 前向传播
h = torch.zeros(1, 1)
hs = []
for t in range(seq_len):
    h = torch.tanh(h @ W_h + x[t:t+1] @ W_x)
    hs.append(h)

# 对最后一个隐藏状态求关于 x[0] 的梯度
h_last = hs[-1]
h_last.backward(retain_graph=True)

# 理论上，梯度 = ∂h_{100}/∂x₀ = ∂h_{100}/∂h_{99} × ∂h_{99}/∂h_{98} × ... × ∂h₁/∂x₀
# 每一步乘一个 tanh 的导数（≤ 1）和 W_h（0.8）

print("=== RNN 梯度消失演示 ===")
print(f"序列长度: {seq_len}")
print(f"W_h = {W_h.item()}")
print(f"tanh 导数最大值 = 1.0")
print(f"每一步梯度最多乘 W_h × tanh'(h) ≤ {W_h.item()}")
print(f"经过 {seq_len} 步后: 梯度 ≤ {W_h.item()}^{seq_len} = {W_h.item()**seq_len:.2e}")
print()
print(f"x[0] 的梯度: {x.grad[0].item():.10f}")
print(f"x[50] 的梯度: {x.grad[50].item():.10f}")
print(f"x[99] 的梯度: {x.grad[99].item():.10f}")
print()
print("结论：序列开头的梯度几乎为 0 → 模型学不到长距离依赖！")

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 左图：W_h 对梯度衰减的影响
wh_values = [0.5, 0.8, 0.95, 1.0]
positions = range(seq_len)
for wh in wh_values:
    gradient_at_step = [wh ** pos for pos in positions]
    ax1.plot(positions, gradient_at_step, label=f'W_h={wh}')
ax1.set_xlabel('序列位置')
ax1.set_ylabel('梯度缩放因子（log scale）')
ax1.set_yscale('log')
ax1.set_title('不同 W_h 下的梯度衰减')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 右图：梯度随位置的实际分布
grads = [x.grad[i].item() for i in range(seq_len)]
ax2.bar(positions, grads, width=1.0)
ax2.set_xlabel('序列位置')
ax2.set_ylabel('梯度大小')
ax2.set_title(f'实际梯度分布 (W_h={W_h.item()})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### 为什么 Attention 是答案？

RNN 的根本问题是：**远距离的信息必须经过中间每一步的传递**，每次传递都有信息损失。

Attention 的解决方案极其简单粗暴：**别传了，直接看！**

```
RNN:  第 1 个词 → 传 → 传 → 传 → ... → 传 → 到达第 100 个词
      （经过 99 次传递，信息衰减殆尽）

Attention: 第 100 个词 —直接看—→ 第 1 个词
           （一步到位，没有衰减！）
```

这不是比喻——Attention 的数学公式里，**每个位置的 Query 和所有位置的 Key 直接做点积**。
无论距离多远，路径长度恒为 O(1)。

而且：
- **并行计算**：所有位置同时算 attention，不需要等前一步的结果（训练时）
- **可解释**：attention 权重直接告诉你「模型在关注什么」
- **长距离友好**：1000 步外的词和旁边的词一样容易注意到

这三个优势直接让 RNN/LSTM 从 NLP 的舞台上退场，Transformer 时代开启。

OK，铺垫够了。下面让我们进入 Attention 的数学世界。

### 1. 从一句话说清 Attention

假设你要翻译：`"The cat sat on the mat"`。

当模型处理到 `"sat"` 时，它需要知道：
- `"sat"` 和 `"cat"` 关系密切（猫在坐）；
- `"sat"` 和 `"mat"` 有关系（坐在垫子上）；
- `"sat"` 和 `"the"` 关系较弱。

**Attention 做的事就是：让每个 token 去「看」所有其他 token，自己决定该关注谁。**

```
      "sat" 关注各 token 的程度：
      
      The  cat  sat  on  the  mat
       ↑    ↑   ↑   ↑   ↑    ↑
      0.05 0.35 0.1 0.05 0.05 0.40
      
      "cat" 得 0.35，"mat" 得 0.40 → sat 主要关注 cat 和 mat
```

**核心公式（先看一遍，等下逐项拆解）：**

```
Attention(Q, K, V) = softmax(Q × K^T / √d) × V
```

Q、K、V 到底是什么？我们用一个小例子手算一遍。

### 2. Q、K、V 的直觉（最重要的一节）

想象你在图书馆找书：

- **Q（Query，查询）**：你想找什么？ → "我想找关于猫的书"
- **K（Key，键）**：每本书的标签 → "这本书讲猫"、"这本讲狗"、"这本讲垫子"
- **V（Value，值）**：每本书的实际内容 → 整本书的正文

过程：
1. 拿你的 Query 和每本书的 Key 做匹配 → 算出每本书和你的需求有多相关
2. 用这个"相关度"做权重，把所有书的 Value 加权平均
3. 结果 = 你实际得到的"混合信息"

**在 Attention 里，每个 token 同时扮演 Q、K、V 三个角色：**
- 作为 Q：我问所有 token 「你们跟我有多相关？」
- 作为 K：其他 token 问我和他们有多相关时，我提供匹配标签
- 作为 V：如果我被选为高相关，我提供我的实际信息

三者都是通过**同一个输入向量 × 三个不同的矩阵**生成的：

### 1.5 从词表到 Embedding 矩阵 — X 到底是怎么来的？

在进入 Q/K/V 之前，我们先搞清楚 **「输入矩阵 X」是怎么从一句人话变出来的**。

完整链路：

```
"the cat sat"
     ↓  词表映射（word → id）
 [2, 5, 8]          ← 3 个 token id
     ↓  Embedding 查表（id → 向量）
 [3, 4]             ← 每个 token 变成一个 4 维向量
     ↓
  X = [[...],        ← 输入矩阵 (3, 4)
       [...],
       [...]]
```

下面一步步来：

In [ ]:
# === 词表 → 句子 → Token IDs → Embedding 查表 → X ===
vocab = {"the": 0, "a": 1, "cat": 2, "dog": 3, "sat": 4,
         "ran": 5, "on": 6, "mat": 7, "[PAD]": 8, "[UNK]": 9}
vocab_size = len(vocab)
id2word = {v: k for k, v in vocab.items()}

# 句子 → token ids
sentence = ["the", "cat", "sat"]
token_ids = [vocab[w] for w in sentence]  # [0, 2, 4]

# token ids → Embedding 查表 → X
d_model = 4
embedding = nn.Embedding(vocab_size, d_model)
token_ids_tensor = torch.tensor(token_ids)  # [3]
X = embedding(token_ids_tensor)             # [3, 4]

print(f"词表大小: {vocab_size}, 句子: {' '.join(sentence)}")
print(f"Token IDs: {token_ids}")
print(f"X 形状: {list(X.shape)}  ← [seq_len=3, d_model=4]")
print(f"\nX = (来自 Embedding 查表，不是 randn):\n{X}")
print(f"\n解释: X[0]='the' → {X[0].tolist()}")
print(f"      X[1]='cat' → {X[1].tolist()}")
print(f"      X[2]='sat' → {X[2].tolist()}")

In [ ]:
# === 从上一步的 X 出发，计算 Q/K/V ===
seq_len = X.shape[0]  # = 3
d_k = 4

# Q/K/V 都来自 X，但各自乘不同的权重矩阵（用 nn.Linear，和后面 MHA 一致）
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)

Q = W_Q(X)  # [3, 4] — 每个 token 的「查询」
K = W_K(X)  # [3, 4] — 每个 token 的「标签」
V = W_V(X)  # [3, 4] — 每个 token 的「内容」

print(f"X 形状: {X.shape}  →  Q/K/V 形状: {Q.shape}")
print(f"→ Q、K、V 都来自同一个 X，乘了不同的矩阵")

### 3. 手算 Attention 分数（一步步来）

**Step 1: 算 Q × K^T — 每个 token 对每个 token 的相关度**

```
第 i 行、第 j 列 = token i 的 Q 和 token j 的 K 做点积
                   → token i 对 token j 的「注意力分数」
```

In [ ]:
# Step 1: 注意力分数 = Q × K^T
# 第 i 行第 j 列 = token i 对 token j 的原始相关度
attention_scores = Q @ K.T  # [3, 4] @ [4, 3] = [3, 3]

print(f"注意力分数矩阵 {list(attention_scores.shape)} = [{seq_len}×{seq_len}]:")
print(attention_scores)
print(f"\n第 i 行 = token {list(range(seq_len))} 对各 token 的分数")

**Step 2: 缩放 — 除以 √d_k**

为什么？因为当 d_k 很大时，点积的结果会变得很大 → softmax 会"过于自信"（梯度消失）。
除以 √d_k 让方差稳定在 1。

In [ ]:
# Step 2: 缩放 / √d_k  → 防止 d_k 大时点积过大，softmax 梯度消失
d_k = Q.shape[-1]
scaled_scores = attention_scores / math.sqrt(d_k)

print(f"缩放因子: √{d_k} = {math.sqrt(d_k):.2f}")
print(f"缩放前 token 0: {attention_scores[0].tolist()}")
print(f"缩放后 token 0: {scaled_scores[0].tolist()}  ← 值变小，相对大小不变")

**Step 3: Softmax — 把分数变成概率（权重）**

softmax 做的事：把一行的所有值变成 0~1 之间的数，加起来等于 1。

```
softmax([2, 3, 1]) → [0.24, 0.67, 0.09]
                加起来刚好等于 1
```

In [ ]:
# Step 3: Softmax → 把分数变成概率（每行加起来 = 1）
attention_weights = F.softmax(scaled_scores, dim=-1)

print(f"注意力权重矩阵 {list(attention_weights.shape)}:")
print(attention_weights)

# 验证每行和为 1
print(f"\n每行和: {attention_weights.sum(dim=-1).tolist()}  ← 都是 1.0")

**Step 4: 加权求和 — 把权重应用到 V 上**

有了「该关注谁」，现在把各路信息加权混合，得到每个 token 的最终输出。

In [ ]:
# Step 4: 加权求和 — 用注意力权重混合 V
output = attention_weights @ V  # [3, 3] @ [3, 4] = [3, 4]

print(f"输出形状: {list(output.shape)} = [{seq_len}, {d_model}]")
print(f"\n输入  token 0: {X[0].tolist()}")
print(f"输出 token 0: {output[0].tolist()}")
print(f"→ 不一样！因为 token 0 按权重融合了 token 1、2 的信息")

### 4. 完整 Attention 的 4 步法总结

```
输入: X (seq_len, d_model)

1. 线性变换: Q = X @ W_Q,  K = X @ W_K,  V = X @ W_V
2. 算分数:   scores = Q @ K^T
3. 缩放:     scores = scores / √d_k
4. Softmax:  weights = softmax(scores, dim=-1)
5. 加权求和: output = weights @ V

输出: out (seq_len, d_model)  ← 每个 token 融合了全局信息
```

这就是 **Scaled Dot-Product Attention**，Transformer 最核心的计算。

### 5. 加入 Mask：为什么需要遮住未来？

训练 LLM 时，我们要让模型**只看到过去**，不能看未来。

比如预测 "cat" 后面的词时，模型不能偷看答案 "sat"。

怎么做？在 softmax 之前，把「未来位置」的分数设为 -∞，softmax 后自然变成 0。

```
原始分数矩阵:          Mask 后:                Softmax 后:
[a, b, c]              [a, -∞, -∞]             [1.0, 0.0, 0.0]
[d, e, f]      →       [d,  e, -∞]      →      [0.4, 0.6, 0.0]
[g, h, i]              [g,  h,  i]             [0.3, 0.3, 0.4]

Token 0 只能看到自己
Token 1 可以看到 0 和 1
Token 2 可以看到 0、1、2
```

In [ ]:
# Causal mask：下三角矩阵，1=允许看，0=遮住（→ -inf）
def causal_mask(seq_len):
    return torch.tril(torch.ones(seq_len, seq_len))

mask = causal_mask(5)
print("Causal Mask (1=允许看, 0=不允许):")
print(mask)
# 第 i 行 = token i 能看到的位置：token 0 只看 [0]，token 2 看 [0,1,2]

### 6. 多头注意力（Multi-Head Attention）

单头注意力只有一组 Q/K/V，只从一个「视角」看问题。

多头注意力 = **多组 Q/K/V 并行计算，然后拼起来**。

```
头 1: 可能学会了「语法关系」（主语-谓语）
头 2: 可能学会了「位置关系」（距离远近）
头 3: 可能学会了「语义关系」（猫-动物）
...
```

每个头只看 d_k = d_model / num_heads 维。多头的总计算量和单头一样，但更灵活。

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    多头因果自注意力
    
    参数:
        d_model: 输入/输出维度
        num_heads: 注意力头数
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # Q、K、V 的线性变换（把 num_heads 个头合并到矩阵操作里）
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        
        # 输出投影
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        """
        输入: x shape = [batch, seq_len, d_model]
        输出:   shape = [batch, seq_len, d_model]
        """
        batch_size, seq_len, _ = x.shape
        
        # 1. 线性变换 + 拆成多头
        #    [batch, seq_len, d_model] → [batch, num_heads, seq_len, d_k]
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. 注意力分数: Q @ K^T
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # 3. Mask（把未来的位置设为 -inf）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # 4. Softmax
        weights = F.softmax(scores, dim=-1)
        
        # 5. 加权求和
        attn_output = weights @ V  # [batch, num_heads, seq_len, d_k]
        
        # 6. 拼回头并投影
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)


In [ ]:
# 测试多头注意力
d_model, num_heads = 8, 2
mha = MultiHeadAttention(d_model, num_heads)

test_x = torch.randn(1, 5, d_model)                         # batch=1, 5 token, 8 维
test_mask = causal_mask(5).unsqueeze(0).unsqueeze(0)        # [1, 1, 5, 5]

out = mha(test_x, test_mask)
print(f"输入: {test_x.shape}  →  输出: {out.shape}  ← 形状不变，内容融合了全局注意力")

### 7. Transformer Block = Attention + FFN + 两个技巧

一个完整的 Transformer Block 包含：

```
输入 x
  ↓
  ├→ Multi-Head Self-Attention ─→ +
  │                               ↓
  └──────────── (残差连接) ────→ [Add & Norm]
                                   ↓
  ├→ Feed-Forward Network ─────→ +
  │                               ↓
  └──────────── (残差连接) ────→ [Add & Norm]
                                   ↓
                                 输出
```

两个关键技巧：
1. **残差连接（Residual）**：输出 = 输入 + 子层输出。让深层网络也能训练。
2. **Layer Normalization（层归一化）**：把每层的输出标准化到均值 0 方差 1，让训练稳定。

In [ ]:
class FeedForward(nn.Module):
    """FFN：两层全连接，先扩 4 倍再压回"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerBlock(nn.Module):
    """一个 Transformer 解码器层：Attention + FFN，各带残差 + LayerNorm"""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        x = self.norm1(x + self.attention(x, mask))  # Attention + 残差 + Norm
        x = self.norm2(x + self.ffn(x))              # FFN + 残差 + Norm
        return x

# 测试
block = TransformerBlock(d_model=8, num_heads=2)
test_out = block(test_x, test_mask)
print(f"TransformerBlock 输入: {test_x.shape}  →  输出: {test_out.shape}  ← 形状不变")

### 8. 完整 Mini GPT 模型

把所有组件拼起来：Embedding → N 层 Transformer Block → 输出投影到词表

In [ ]:
# 复用 Part 3 的位置编码
def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class MiniGPT(nn.Module):
    """Mini GPT: Embedding → N×TransformerBlock → LayerNorm → 投影到词表"""
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=128):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = get_sinusoidal_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)  # 投影到词表 → logits
    
    def forward(self, x):
        # x: [batch, seq_len]  token IDs
        batch_size, seq_len = x.shape
        x = self.token_emb(x) + self.pe[:seq_len, :]          # Embedding + Position
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        mask = mask.view(1, 1, seq_len, seq_len)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_final(x)
        return self.lm_head(x)  # [batch, seq_len, vocab_size]

In [ ]:
# 测试 MiniGPT
vocab_size = 30
model = MiniGPT(vocab_size=vocab_size, d_model=64, num_heads=4, num_layers=2)

total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}, 可训练: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 前向传播: batch=2, seq_len=8
batch_input = torch.randint(0, vocab_size, (2, 8))
logits = model(batch_input)
print(f"\n输入: {batch_input.shape}  →  输出: {logits.shape}")
print(f"输出 = [batch=2, seq=8, vocab={vocab_size}] → 每个位置对每个词的 logits")
print(f"logits[0, 7, :5]: {logits[0, 7, :5].tolist()}  ← 样本 0 位置 7 预测下一 token 的分数")

### 9. 模型的输入输出全貌

现在我们可以画一张完整的数据流图：

```
                 "the cat sat on the mat"
                         ↓ Tokenizer
                 [3, 7, 15, 2, 3, 12]
                         ↓ Token Embedding + Position
               [batch, 6, 64]  ← 6 个 token，每个 64 维向量
                         ↓ N × Transformer Block
               [batch, 6, 64]  ← 形状不变，信息深度融合
                         ↓ LayerNorm + Linear
               [batch, 6, 30]  ← 每个位置 30 个分数
                                  ↑
                            30 = 词表大小
```

**最后这 30 个分数就是 logits**。
把它们送进 softmax → 变成概率 → 概率最大的就是模型预测的下一个 token。

但现在模型还是随机参数，预测出来的东西是垃圾。
怎么训练？Loss 怎么算？

→ **下一个 Part：训练循环与 Loss 构建。** 这是你最关心的部分。

---

## 第四部分小结

确认你已经搞懂了这些（按顺序检查）：

1. ✅ Attention 的核心：让每个 token 去看所有 token，自己决定该关注谁
2. ✅ Q=查询（我想找什么），K=标签（我是什么），V=内容（我有什么信息）
3. ✅ Q、K、V 都来自同一个 X，乘不同的矩阵
4. ✅ 注意力分数 = Q × K^T → 缩放 → softmax → 得到权重
5. ✅ 输出 = 权重 × V，融合了全局信息
6. ✅ Causal mask 确保不看未来（训练时需要）
7. ✅ 多头 = 多组并行的 Q/K/V，从不同视角看
8. ✅ Transformer Block = Attention + FFN + 残差 + LayerNorm
9. ✅ MiniGPT = Embedding + N × Block + 输出投影

**如果你对 Q/K/V 有任何模糊，请回去重看第 2、3 节的手算例子。**

全部清楚了？→ 下一步，训练！Loss 到底怎么算！